# Fase 1 - Exploração: Seleção de Mercados

Este notebook explora a seleção de mercados da Polymarket para o backtesting.

## Objetivos:
1. Carregar mercados resolvidos via API
2. Filtrar por critérios (volume, duração, tipo binário)
3. Dividir em grupos A/B/C
4. Analisar distribuição de mercados

In [ ]:
# Setup do path para imports
import sys
sys.path.insert(0, '..')

# Imports
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from config.settings import (
    MIN_VOLUME_USD,
    MIN_LIFETIME_DAYS,
    TOP_N_LARGE_MARKETS,
    ensure_data_dirs_exist,
)
from core.api_client import get_api_client
from pipeline.phase1_market_selection import (
    load_all_resolved_markets,
    filter_markets_by_criteria,
    split_markets_into_ABC,
    load_markets_from_json,
    get_market_summary,
)

# Configuração de visualização
plt.style.use('seaborn-v0_8-whitegrid')
pd.set_option('display.max_columns', 20)
pd.set_option('display.width', 200)

ensure_data_dirs_exist()

## 1. Carregar Mercados da API

In [ ]:
# Carrega todos os mercados resolvidos
# ATENÇÃO: Isso pode demorar alguns minutos
api_client = get_api_client()
all_markets = load_all_resolved_markets(api_client, max_pages=10)  # Limitar para teste

print(f"Total de mercados carregados: {len(all_markets)}")

In [ ]:
# Converte para DataFrame para análise
markets_df = pd.DataFrame([m.to_dict() for m in all_markets])
markets_df.head()

## 2. Análise Exploratória

In [ ]:
# Estatísticas básicas de volume
print("Estatísticas de Volume (USD):")
print(markets_df['volume'].describe())

In [ ]:
# Distribuição de volume (histograma)
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Histograma normal
axes[0].hist(markets_df['volume'], bins=50, edgecolor='black')
axes[0].set_xlabel('Volume (USD)')
axes[0].set_ylabel('Frequência')
axes[0].set_title('Distribuição de Volume')

# Histograma log scale
axes[1].hist(markets_df['volume'][markets_df['volume'] > 0], bins=50, edgecolor='black')
axes[1].set_xlabel('Volume (USD)')
axes[1].set_ylabel('Frequência')
axes[1].set_xscale('log')
axes[1].set_title('Distribuição de Volume (Log Scale)')

plt.tight_layout()
plt.show()

In [ ]:
# Distribuição por categoria
category_counts = markets_df['category'].value_counts().head(15)

plt.figure(figsize=(12, 6))
category_counts.plot(kind='bar')
plt.xlabel('Categoria')
plt.ylabel('Número de Mercados')
plt.title('Top 15 Categorias de Mercados')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()

## 3. Filtrar por Critérios

In [ ]:
# Aplica filtros
filtered_markets = filter_markets_by_criteria(
    all_markets,
    volume_min=MIN_VOLUME_USD,
    min_lifetime_days=MIN_LIFETIME_DAYS,
    binary_only=True,
)

print(f"Mercados após filtro: {len(filtered_markets)} de {len(all_markets)}")
print(f"Taxa de retenção: {len(filtered_markets)/len(all_markets)*100:.1f}%")

## 4. Dividir em Grupos A/B/C

In [ ]:
# Divide em grupos
groups = split_markets_into_ABC(filtered_markets)

for group_name, markets in groups.items():
    summary = get_market_summary(markets)
    print(f"\nGrupo {group_name}: {summary['count']} mercados")
    if summary['count'] > 0:
        vol = summary['volume']
        print(f"  Volume total: ${vol['total']:,.0f}")
        print(f"  Volume range: ${vol['min']:,.0f} - ${vol['max']:,.0f}")

In [ ]:
# Visualização da divisão de grupos
group_data = {
    'Grupo': [],
    'Mercados': [],
    'Volume Total': [],
}

for group_name, markets in groups.items():
    group_data['Grupo'].append(group_name)
    group_data['Mercados'].append(len(markets))
    group_data['Volume Total'].append(sum(m.volume for m in markets))

group_df = pd.DataFrame(group_data)

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Número de mercados por grupo
axes[0].bar(group_df['Grupo'], group_df['Mercados'], color=['#2ecc71', '#3498db', '#e74c3c'])
axes[0].set_xlabel('Grupo')
axes[0].set_ylabel('Número de Mercados')
axes[0].set_title('Mercados por Grupo')

# Volume por grupo
axes[1].bar(group_df['Grupo'], group_df['Volume Total'] / 1e6, color=['#2ecc71', '#3498db', '#e74c3c'])
axes[1].set_xlabel('Grupo')
axes[1].set_ylabel('Volume Total (Milhões USD)')
axes[1].set_title('Volume por Grupo')

plt.tight_layout()
plt.show()

## 5. Exemplos de Mercados por Grupo

In [ ]:
# Mostra exemplos de cada grupo
for group_name, markets in groups.items():
    print(f"\n=== Grupo {group_name} (Top 5) ===")
    for market in markets[:5]:
        print(f"  [{market.id[:8]}...] ${market.volume:,.0f} - {market.question[:60]}...")

## 6. Próximos Passos

Com os mercados selecionados e agrupados, podemos prosseguir para:

1. **Fase 2**: Coleta de séries históricas de preço YES/NO
2. **Fase 3**: Análise de spreads e oportunidades de arbitragem
3. **Fase 4**: Estatísticas agregadas